# Task 8 — Flow Assignment

Assign 2025 and 2030 county-to-county freight flows through the multi-tier network; compute throughput at every node and every link; identify critical hubs and corridors.

## Core simplifying assumptions

- Freight routes through the **nearest assigned hub** in the hierarchy (no multi-hop shortest-path routing).
- For multi-gateway areas: flow split proportionally by `usable_available_space_sf` share across gateways.
- For multi-hub regions (0 and 7): flow split proportionally by `usable_available_space_sf` share across the 2 assigned hubs — applied consistently in **both** 8.3 and 8.4.
- Interface node flows use Task 2 pre-allocated tons directly as boundary conditions (no re-derivation).
- Commodity filter: exclude `sctg1014` (gravel) and `sctg1519` (coal/energy) — same as Task 5.
- **Scope boundary:** NE-internal flows route through gateway and hub tiers. NE-external flows credited directly to nearest regional hub in 8.6 — gateways do not see external traffic.
- **`external_throughput_ktons` naming collision:** `region_metrics.csv` / `area_metrics_phase2.csv` store NE-to-non-NE boundary flows only (447,344 ktons). `region_external_metrics.csv` stores all inter-region flows including NE-NE cross-region traffic (1,829,087 ktons). Never mix these two sources.

## 8.1 — County Routing Lookup Table

Build a flat lookup table mapping every NE county to its gateway(s) and regional hub(s) with capacity-share weights.

**Routing share decomposition:**

$$
\text{combined\_share}_{c,g,h} = \underbrace{\frac{s_g}{\sum_{g' \in A(c)} s_{g'}}}_{\text{gw\_share}} \times \underbrace{\frac{s_h}{\sum_{h' \in R(c)} s_{h'}}}_{\text{hub\_share}}
$$

where $s_g$, $s_h$ are `usable_available_space_sf` for gateway $g$ and hub $h$, $A(c)$ is the area containing county $c$, and $R(c)$ is the region containing county $c$.

**Assert:** $\sum_{(g,h)} \text{combined\_share}_{c,g,h} = 1.0$ for every county $c$.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

OUT_DIR = Path("../Data/Task8")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# ── Step 1: County → area join ─────────────────────────────────────────────
area_assign = pd.read_csv("../Data/Task6/area_assignment.csv",
                          dtype={"fips": "int64", "area_id": "str", "region_id": "int64"})
# keep only the columns we need
county_area = area_assign[["fips", "area_id", "region_id"]].copy()

print(f"Step 1 — county_area: {len(county_area)} rows | "
      f"unique fips: {county_area['fips'].nunique()} | "
      f"unique areas: {county_area['area_id'].nunique()} | "
      f"unique regions: {county_area['region_id'].nunique()}")

Step 1 — county_area: 434 rows | unique fips: 434 | unique areas: 132 | unique regions: 50


In [2]:
# ── Step 2: Area → gateway(s) with capacity shares ─────────────────────────
gw_assign = pd.read_csv("../Data/Task6/gateway_area_assignments.csv",
                        dtype={"candidate_id": "str", "area_id": "str",
                               "usable_available_space_sf": "int64"})
gw_raw = gw_assign[["candidate_id", "area_id", "usable_available_space_sf"]].copy()

# gw_sqft_total[a] = sum(sqft) for all gateways in area a
gw_area_total = gw_raw.groupby("area_id")["usable_available_space_sf"].sum().rename("gw_sqft_total")
gw_shares = gw_raw.join(gw_area_total, on="area_id")
gw_shares["gw_share"] = gw_shares["usable_available_space_sf"] / gw_shares["gw_sqft_total"]
gw_shares = gw_shares[["candidate_id", "area_id", "gw_share"]].rename(
    columns={"candidate_id": "gateway_candidate_id"})

print(f"Step 2 — gw_shares: {len(gw_shares)} rows | "
      f"unique areas: {gw_shares['area_id'].nunique()} | "
      f"unique gateways: {gw_shares['gateway_candidate_id'].nunique()}")
print(f"  gw_share range: [{gw_shares['gw_share'].min():.6f}, {gw_shares['gw_share'].max():.6f}]")

# Spot-check: per-area sums should all be 1.0
area_gw_sum = gw_shares.groupby("area_id")["gw_share"].sum()
bad = area_gw_sum[~np.isclose(area_gw_sum, 1.0, atol=1e-9)]
assert len(bad) == 0, f"gw_share does not sum to 1.0 in areas: {bad.index.tolist()}"
print(f"  ✓ gw_share sums to 1.0 in all {area_gw_sum.shape[0]} areas")

Step 2 — gw_shares: 319 rows | unique areas: 132 | unique gateways: 312
  gw_share range: [0.146180, 1.000000]
  ✓ gw_share sums to 1.0 in all 132 areas


In [3]:
# ── Step 3: Region → hub(s) with capacity shares ───────────────────────────
hub_assign = pd.read_csv("../Data/Task5/hub_region_assignments.csv",
                         dtype={"candidate_id": "str", "region_id": "int64",
                                "usable_available_space_sf": "int64"})
hub_raw = hub_assign[["candidate_id", "region_id", "usable_available_space_sf"]].copy()

# hub_sqft_total[r] = sum(sqft) for all hubs in region r
hub_region_total = hub_raw.groupby("region_id")["usable_available_space_sf"].sum().rename("hub_sqft_total")
hub_shares = hub_raw.join(hub_region_total, on="region_id")
hub_shares["hub_share"] = hub_shares["usable_available_space_sf"] / hub_shares["hub_sqft_total"]
hub_shares = hub_shares[["candidate_id", "region_id", "hub_share"]].rename(
    columns={"candidate_id": "hub_candidate_id"})

print(f"Step 3 — hub_shares: {len(hub_shares)} rows | "
      f"unique regions: {hub_shares['region_id'].nunique()} | "
      f"unique hubs: {hub_shares['hub_candidate_id'].nunique()}")

# Identify multi-hub regions
multi_hub = hub_shares.groupby("region_id").size()
multi_hub_regions = multi_hub[multi_hub > 1]
print(f"  Multi-hub regions: {multi_hub_regions.to_dict()}")

# Spot-check: per-region sums should all be 1.0
region_hub_sum = hub_shares.groupby("region_id")["hub_share"].sum()
bad = region_hub_sum[~np.isclose(region_hub_sum, 1.0, atol=1e-9)]
assert len(bad) == 0, f"hub_share does not sum to 1.0 in regions: {bad.index.tolist()}"
print(f"  ✓ hub_share sums to 1.0 in all {region_hub_sum.shape[0]} regions")

Step 3 — hub_shares: 52 rows | unique regions: 50 | unique hubs: 50
  Multi-hub regions: {0: 2, 7: 2}
  ✓ hub_share sums to 1.0 in all 50 regions


In [4]:
# ── Step 4: Assemble flat routing table ────────────────────────────────────
# county_area  : fips → (area_id, region_id)
# gw_shares    : area_id → (gateway_candidate_id, gw_share)
# hub_shares   : region_id → (hub_candidate_id, hub_share)
#
# Join chain:
#   county_area
#     LEFT JOIN gw_shares  ON area_id          → (county, area, region, gateway, gw_share)
#     LEFT JOIN hub_shares ON region_id         → (county, area, region, gateway, hub, gw_share, hub_share)
#   combined_share = gw_share × hub_share

# county × gateway  (via area)
county_gw = county_area.merge(gw_shares, on="area_id", how="left")

# check for any counties that got no gateways
no_gw = county_gw[county_gw["gateway_candidate_id"].isna()]
if len(no_gw) > 0:
    print(f"WARNING: {no_gw['fips'].nunique()} counties have no gateway assignment:")
    print(no_gw["fips"].unique())
else:
    print(f"  ✓ All {county_area['fips'].nunique()} counties have at least one gateway")

# county × gateway × hub  (via region)
routing = county_gw.merge(hub_shares, on="region_id", how="left")

# check for any rows that got no hub
no_hub = routing[routing["hub_candidate_id"].isna()]
if len(no_hub) > 0:
    print(f"WARNING: {no_hub['fips'].nunique()} counties have no hub assignment:")
    print(no_hub["fips"].unique())
else:
    print(f"  ✓ All counties × gateways have a hub assignment")

# combined share
routing["combined_share"] = routing["gw_share"] * routing["hub_share"]

# Final column selection and ordering
lookup = routing[["fips", "area_id", "region_id",
                   "gateway_candidate_id", "hub_candidate_id",
                   "gw_share", "hub_share", "combined_share"]].copy()

print(f"\nStep 4 — lookup: {len(lookup):,} rows | "
      f"unique (fips, gateway, hub) combos: {lookup.drop_duplicates(['fips','gateway_candidate_id','hub_candidate_id']).shape[0]:,}")
print(lookup.head(5).to_string())

  ✓ All 434 counties have at least one gateway
  ✓ All counties × gateways have a hub assignment

Step 4 — lookup: 1,112 rows | unique (fips, gateway, hub) combos: 1,112
    fips area_id  region_id gateway_candidate_id hub_candidate_id  gw_share  hub_share  combined_share
0  42003     0_0          0          T4-PA-00068      T4-PA-00319  0.213927   0.540936        0.115721
1  42003     0_0          0          T4-PA-00068      T4-PA-00350  0.213927   0.459064        0.098206
2  42003     0_0          0          T4-PA-00090      T4-PA-00319  0.167585   0.540936        0.090653
3  42003     0_0          0          T4-PA-00090      T4-PA-00350  0.167585   0.459064        0.076932
4  42003     0_0          0          T4-PA-00329      T4-PA-00319  0.230964   0.540936        0.124937


In [5]:
# ── Sanity check: combined_share sums to 1.0 per fips ─────────────────────
fips_sum = lookup.groupby("fips")["combined_share"].sum()

n_counties = fips_sum.shape[0]
bad_fips = fips_sum[~np.isclose(fips_sum, 1.0, atol=1e-9)]

if len(bad_fips) > 0:
    print(f"FAIL: combined_share does not sum to 1.0 for {len(bad_fips)} counties:")
    print(bad_fips.sort_values())
    raise AssertionError("combined_share sanity check failed — investigate join before proceeding")

print(f"✓ combined_share sums to 1.0 (±1e-9) for all {n_counties} counties")
print(f"  fips_sum stats: min={fips_sum.min():.10f}  max={fips_sum.max():.10f}  "
      f"mean={fips_sum.mean():.10f}")
print(f"  Routing table shape: {lookup.shape}")
print(f"  Rows per county (min/median/max): "
      f"{lookup.groupby('fips').size().min()} / "
      f"{lookup.groupby('fips').size().median():.0f} / "
      f"{lookup.groupby('fips').size().max()}")

✓ combined_share sums to 1.0 (±1e-9) for all 434 counties
  fips_sum stats: min=1.0000000000  max=1.0000000000  mean=1.0000000000
  Routing table shape: (1112, 8)
  Rows per county (min/median/max): 1 / 2 / 10


In [6]:
# ── Save output ────────────────────────────────────────────────────────────
out_path = OUT_DIR / "county_routing_lookup.parquet"
lookup.to_parquet(out_path, index=False)
print(f"Saved → {out_path}  ({out_path.stat().st_size / 1024:.1f} KB)")

Saved → ../Data/Task8/county_routing_lookup.parquet  (19.5 KB)


## Milestone — 8.1 complete

**Output:** `Data/Task8/county_routing_lookup.parquet`

| Column | Type | Description |
|--------|------|-------------|
| `fips` | int64 | County FIPS |
| `area_id` | str | Area assignment |
| `region_id` | int64 | Region assignment |
| `gateway_candidate_id` | str | Gateway hub CoStar ID |
| `hub_candidate_id` | str | Regional hub CoStar ID |
| `gw_share` | float | Share of county flow to this gateway (sums to 1.0 within fips) |
| `hub_share` | float | Share of area flow to this hub (sums to 1.0 within area_id) |
| `combined_share` | float | `gw_share × hub_share` — fraction of county flow reaching (gateway, hub) pair; **asserted to sum to 1.0 per fips** |

**Ready for:** Task 8.2 (area-pair flow matrix) and Task 8.5 (gateway throughput).